# By using lorentzian function we can get a clean spectra looking function in which we add randomness and noise in order to create entry for our dataset

In [3]:
import os
import re
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ─────────────────────────────────────────────────────────────────────────────
# HELPER FUNKCIE
# ─────────────────────────────────────────────────────────────────────────────

def slugify(name: str) -> str:
    """Premení názov zlúčeniny na bezpečný názov priečinka."""
    name = name.strip().lower()
    name = re.sub(r"\s+", "_", name)
    name = re.sub(r"[^a-z0-9_\-\.]", "", name)
    return name or "unknown"


def lorentzian(x: np.ndarray, center: float, hwhm: float) -> np.ndarray:
    """
    Lorentzova funkcia — tvar jedného NMR píku.

    Parametre:
        x      : os ppm hodnôt
        center : poloha píku v ppm
        hwhm   : pološírka píku (Half Width at Half Maximum) v ppm
    """
    return (hwhm ** 2) / ((x - center) ** 2 + hwhm ** 2)


def resample_to(vector: np.ndarray, target_len: int = 1024) -> np.ndarray:
    """Resampľuje vektor na požadovanú dĺžku pomocou lineárnej interpolácie."""
    x_old = np.linspace(0, 1, len(vector))
    x_new = np.linspace(0, 1, target_len)
    return np.interp(x_new, x_old, vector).astype(np.float32)


# ─────────────────────────────────────────────────────────────────────────────
# J-COUPLING
# ─────────────────────────────────────────────────────────────────────────────

def get_multiplet_lines(
    center_ppm:       float,
    n_neighbors:      int,
    j_hz:             float,
    spectrometer_mhz: float = 400.0,
):
    """
    Vypočíta polohy a relatívne intenzity čiar multipletu.

    Čo je J-coupling?
        Susedné H atómy na molekule navzájom ovplyvňujú svoje signály.
        Výsledkom je rozštiepenie píku na multiplet:
            0 susedov → singlet  [1]
            1 sused   → dublet   [1, 1]
            2 susedia → triplet  [1, 2, 1]
            3 susedia → kvartet  [1, 3, 3, 1]
            atď. podľa Pascalovho trojuholníka

    Parametre:
        center_ppm       : stredná poloha píku v ppm
        n_neighbors      : počet susedných H atómov (z CSV)
        j_hz             : veľkosť J-couplingu v Hz
        spectrometer_mhz : frekvencia spektrometra v MHz
    """

    pascal = [
        [1],
        [1, 1],
        [1, 2, 1],
        [1, 3, 3, 1],
        [1, 4, 6, 4, 1],
        [1, 5, 10, 10, 5, 1],
    ]

    n = min(n_neighbors, len(pascal) - 1)
    relative_intensities = np.array(pascal[n], dtype=np.float32)
    relative_intensities /= relative_intensities.sum()

    n_lines     = len(relative_intensities)
    spacing_ppm = j_hz / spectrometer_mhz

    offsets   = np.linspace(-(n_lines - 1) / 2 * spacing_ppm,
                             (n_lines - 1) / 2 * spacing_ppm, n_lines)
    positions = center_ppm + offsets

    return positions, relative_intensities


# ─────────────────────────────────────────────────────────────────────────────
# GENERÁTOR SPEKTRA — čistá zlúčenina
# ─────────────────────────────────────────────────────────────────────────────

def generate_spectrum(
    peaks_ppm:       np.ndarray,
    peaks_int:       np.ndarray,
    peaks_neighbors: np.ndarray,
    ppm_min:         float = -0.5,
    ppm_max:         float = 10.5,
    n_points:        int   = 8192,
    rng:             np.random.Generator | None = None,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Vygeneruje jedno syntetické 1H NMR spektrum jednej zlúčeniny.

    Postup:
        1. Vytvorí os ppm (x)
        2. Pre každý pík zo vstupného CSV:
            a. Pridá malý náhodný shift (simuluje chybu referencovania)
            b. Rozštiepí pík na multiplet podľa n_neighbors z CSV
            c. Každú čiaru multipletu nakreslí ako Lorentzián
        3. Pridá kontaminanty (voda, CDCl3)
        4. Pridá hladký baseline drift
        5. Pridá gaussovský šum
        6. Normalizuje výsledok na rozsah [-1, 1]
    """

    rng = np.random.default_rng() if rng is None else rng

    # ── 1. Os ppm ────────────────────────────────────────────────────────────
    x = np.linspace(ppm_min, ppm_max, n_points).astype(np.float32)
    y = np.zeros_like(x)

    # ── 2. Globálne náhodné variácie ─────────────────────────────────────────
    global_shift = rng.uniform(-0.03, 0.03)   # simuluje chybu referencovania
    global_scale = rng.uniform(0.6,   1.4)    # simuluje rôznu koncentráciu vzorky

    # ── 3. Píky s J-couplingom ───────────────────────────────────────────────
    for peak_ppm, peak_int, n_neighbors in zip(peaks_ppm, peaks_int, peaks_neighbors):

        shifted_ppm = peak_ppm + global_shift + rng.normal(0, 0.002)
        scaled_int  = peak_int * global_scale * rng.uniform(0.85, 1.15)
        hwhm        = rng.uniform(0.0015, 0.006)

        # J-hz len ak má pík susedov (nie singlet)
        j_hz = rng.uniform(3.0, 10.0) if n_neighbors > 0 else 0.0

        line_positions, line_intensities = get_multiplet_lines(
            center_ppm       = shifted_ppm,
            n_neighbors      = int(n_neighbors),
            j_hz             = j_hz,
            spectrometer_mhz = 400.0,
        )

        for line_ppm, line_rel_int in zip(line_positions, line_intensities):
            y += scaled_int * line_rel_int * lorentzian(x, line_ppm, hwhm)

    # ── 4. Kontaminanty ──────────────────────────────────────────────────────
    if rng.random() < 0.7:
        water_ppm  = 4.70 + rng.normal(0, 0.02)
        water_int  = rng.uniform(0.05, 0.2)
        water_hwhm = rng.uniform(0.002, 0.02)
        y += water_int * lorentzian(x, water_ppm, water_hwhm)

    if rng.random() < 0.7:
        cdcl3_ppm  = 7.26 + rng.normal(0, 0.01)
        cdcl3_int  = rng.uniform(0.02, 0.15)
        cdcl3_hwhm = rng.uniform(0.001, 0.01)
        y += cdcl3_int * lorentzian(x, cdcl3_ppm, cdcl3_hwhm)

    # ── 5. Baseline drift ────────────────────────────────────────────────────
    baseline_amplitude = rng.uniform(0.0, 0.003)
    raw_noise          = rng.normal(0, 1, n_points)
    smoothing_kernel   = np.ones(200) / 200
    smooth_baseline    = np.convolve(raw_noise, smoothing_kernel, mode="same")
    y += baseline_amplitude * smooth_baseline

    # ── 6. Gaussovský šum ────────────────────────────────────────────────────
    noise_sigma = rng.uniform(0.00005, 0.0005)
    y += rng.normal(0, noise_sigma, size=n_points).astype(np.float32)

    # ── 7. Normalizácia na [-1, 1] ───────────────────────────────────────────
    y = y - np.median(y)
    p995 = np.percentile(np.abs(y), 99.5)
    y    = np.clip(y, -p995, p995)
    y    = y / (np.max(np.abs(y)) + 1e-6)

    return x, y.astype(np.float32)


# ─────────────────────────────────────────────────────────────────────────────
# GENERÁTOR ZMESI — viacero zlúčenín naraz (multi-label)
# ─────────────────────────────────────────────────────────────────────────────

def generate_mixture(
    all_compounds: dict,
    n_components:  int   = 2,
    ppm_min:       float = -0.5,
    ppm_max:       float = 10.5,
    n_points:      int   = 8192,
    rng:           np.random.Generator | None = None,
) -> tuple[np.ndarray, np.ndarray, list[str]]:
    """
    Vygeneruje spektrum zmesi viacerých zlúčenín.

    Princíp:
        spektrum_zmesi = spektrum_A * ratio_A + spektrum_B * ratio_B + ...

    Každá zlúčenina dostane náhodný mixing ratio (koľko jej je v zmesi).
    Výsledné spektrum je normalizované rovnako ako čisté spektrum.

    Parametre:
        all_compounds : slovník {nazov: (peaks_ppm, peaks_int, peaks_neighbors)}
        n_components  : koľko zlúčenín bude v zmesi (2 alebo 3)
        ppm_min/max   : rozsah spektra
        n_points      : rozlíšenie
        rng           : random generator

    Vracia:
        x          : os ppm
        y          : spektrum zmesi
        components : zoznam názvov zlúčenín v zmesi
    """

    rng = np.random.default_rng() if rng is None else rng

    compound_names = list(all_compounds.keys())
    chosen = rng.choice(compound_names, size=n_components, replace=False).tolist()

    x       = np.linspace(ppm_min, ppm_max, n_points).astype(np.float32)
    y_mixed = np.zeros(n_points, dtype=np.float32)

    # Dirichlet distribúcia zabezpečí že súčet ratio = 1.0
    ratios = rng.dirichlet(np.ones(n_components)).astype(np.float32)

    for compound_name, ratio in zip(chosen, ratios):
        peaks_ppm, peaks_int, peaks_neighbors = all_compounds[compound_name]

        _, y_single = generate_spectrum(
            peaks_ppm       = peaks_ppm,
            peaks_int       = peaks_int,
            peaks_neighbors = peaks_neighbors,
            ppm_min         = ppm_min,
            ppm_max         = ppm_max,
            n_points        = n_points,
            rng             = rng,
        )

        y_mixed += ratio * y_single

    # Normalizácia zmesi
    y_mixed = y_mixed - np.median(y_mixed)
    p995    = np.percentile(np.abs(y_mixed), 99.5)
    y_mixed = np.clip(y_mixed, -p995, p995)
    y_mixed = y_mixed / (np.max(np.abs(y_mixed)) + 1e-6)

    return x, y_mixed.astype(np.float32), chosen


# ─────────────────────────────────────────────────────────────────────────────
# ULOŽENIE PNG
# ─────────────────────────────────────────────────────────────────────────────

def save_png(x: np.ndarray, y: np.ndarray, path_png: str, dpi: int = 100, linewidth: float = 4.0):
    """
    Uloží spektrum ako PNG obrázok.

    Dôležité detaily:
        - figsize=(10.24, 2.46) pri dpi=100 → presne 1024x246 pixelov
        - invert_xaxis: NMR konvencia — ppm rastie sprava doľava
        - subplots_adjust: graf vyplní celú plochu bez okrajov
        - Pevné y limity: zabezpečia konzistenciu pri CV2 extrakcii
    """
    fig, ax = plt.subplots(figsize=(10.24, 2.46))
    ax.plot(x, y, linewidth=linewidth, color="black")
    ax.invert_xaxis()
    ax.set_ylim(-0.2, 1.1)
    ax.set_xlim(x.min(), x.max())
    ax.axis("off")                   # opravené z "x" na "off"
    fig.patch.set_facecolor("white")
    fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
    plt.savefig(path_png, dpi=dpi, facecolor="white")
    plt.close()


# ─────────────────────────────────────────────────────────────────────────────
# HLAVNÝ BUILDER
# ─────────────────────────────────────────────────────────────────────────────

def build_dataset(
    csv_path:             str,
    out_dir:              str   = "dataset_out",
    samples_per_compound: int   = 500,
    save_vectors:         bool  = True,
    save_images:          bool  = True,
    vector_len:           int   = 1024,
    generate_mixtures:    bool  = True,
    mixture_ratio:        float = 0.3,
    max_components:       int   = 3,
    seed:                 int   = 0,
):
    """
    Vygeneruje celý dataset zo vstupného CSV súboru.

    Štruktúra výstupu:
        out_dir/
        ├── label_map.json   ← mapovanie názov → index
        ├── index.csv        ← zoznam všetkých vzoriek s labelmi
        ├── images/
        │   ├── acetone/
        │   │   ├── acetone_00000.png
        │   │   └── ...
        │   └── _mixtures/
        │       ├── mixture_00000.png
        │       └── ...
        └── vectors/
            ├── acetone/
            │   ├── acetone_00000.npy
            │   └── ...
            └── _mixtures/
                ├── mixture_00000.npy
                └── ...

    Pre čisté zlúčeniny: label je číslo (0–41)
    Pre zmesi: label je zoznam čísel napr. "[0, 5]" pre acetón + benzén

    CSV musí obsahovať stĺpce: compound, ppm, intensity, n_neighbors
    """

    rng = np.random.default_rng(seed)
    df  = pd.read_csv(csv_path)

    # Validácia
    required_columns = {"compound", "ppm", "intensity", "n_neighbors"}
    if not required_columns.issubset(df.columns):
        raise ValueError(f"CSV musí obsahovať: {required_columns} (nájdené: {set(df.columns)})")

    df = df.dropna(subset=["compound", "ppm", "intensity", "n_neighbors"]).copy()
    df["compound"] = df["compound"].astype(str)

    os.makedirs(out_dir, exist_ok=True)

    # Label mapa
    compounds = sorted(df["compound"].unique().tolist())
    label_map = {compound: idx for idx, compound in enumerate(compounds)}

    with open(os.path.join(out_dir, "label_map.json"), "w", encoding="utf-8") as f:
        json.dump(label_map, f, ensure_ascii=False, indent=2)

    # Priprav slovník všetkých zlúčenín pre mixture generátor
    all_compounds = {}
    for compound, group in df.groupby("compound"):
        all_compounds[compound] = (
            group["ppm"].to_numpy(np.float32),
            group["intensity"].to_numpy(np.float32),
            group["n_neighbors"].to_numpy(int),
        )

    # Vytvorenie výstupných priečinkov
    if save_vectors:
        os.makedirs(os.path.join(out_dir, "vectors"), exist_ok=True)
        if generate_mixtures:
            os.makedirs(os.path.join(out_dir, "vectors", "_mixtures"), exist_ok=True)
    if save_images:
        os.makedirs(os.path.join(out_dir, "images"), exist_ok=True)
        if generate_mixtures:
            os.makedirs(os.path.join(out_dir, "images", "_mixtures"), exist_ok=True)

    rows = []

    # ── Generuj čisté zlúčeniny ───────────────────────────────────────────────
    for compound, group in df.groupby("compound", sort=False):

        peaks_ppm       = group["ppm"].to_numpy(np.float32)
        peaks_int       = group["intensity"].to_numpy(np.float32)
        peaks_neighbors = group["n_neighbors"].to_numpy(int)
        folder_name     = slugify(compound)

        vec_dir = os.path.join(out_dir, "vectors", folder_name)
        img_dir = os.path.join(out_dir, "images",  folder_name)

        if save_vectors: os.makedirs(vec_dir, exist_ok=True)
        if save_images:  os.makedirs(img_dir, exist_ok=True)

        # Ak generujeme zmesi, čistých vzoriek je menej
        n_pure = int(samples_per_compound * (1 - mixture_ratio)) if generate_mixtures else samples_per_compound

        for i in range(n_pure):
            x, y = generate_spectrum(peaks_ppm, peaks_int, peaks_neighbors, rng=rng)

            y_final = resample_to(y, target_len=vector_len)
            x_final = np.linspace(x.min(), x.max(), vector_len)

            sample_id = f"{folder_name}_{i:05d}"
            png_path  = ""
            npy_path  = ""

            if save_images:
                png_path = os.path.join(img_dir, f"{sample_id}.png")
                save_png(x_final, y_final, png_path)

            if save_vectors:
                npy_path = os.path.join(vec_dir, f"{sample_id}.npy")
                np.save(npy_path, y_final)

            rows.append({
                "sample_id":  sample_id,
                "compound":   compound,
                "label":      label_map[compound],
                "is_mixture": False,
                "components": compound,
                "png_path":   png_path,
                "npy_path":   npy_path,
            })

        print(f"Čisté: {compound} → {n_pure} vzoriek")

    # ── Generuj zmesi ─────────────────────────────────────────────────────────
    if generate_mixtures:
        n_mixtures = int(samples_per_compound * mixture_ratio * len(compounds))
        mix_vec_dir = os.path.join(out_dir, "vectors", "_mixtures")
        mix_img_dir = os.path.join(out_dir, "images",  "_mixtures")

        for i in range(n_mixtures):
            n_components = rng.integers(2, max_components + 1)

            x, y_mix, chosen = generate_mixture(
                all_compounds = all_compounds,
                n_components  = n_components,
                rng           = rng,
            )

            y_final   = resample_to(y_mix, target_len=vector_len)
            x_final   = np.linspace(x.min(), x.max(), vector_len)
            sample_id = f"mixture_{i:05d}"
            png_path  = ""
            npy_path  = ""

            if save_images:
                png_path = os.path.join(mix_img_dir, f"{sample_id}.png")
                save_png(x_final, y_final, png_path)

            if save_vectors:
                npy_path = os.path.join(mix_vec_dir, f"{sample_id}.npy")
                np.save(npy_path, y_final)

            component_labels = [label_map[c] for c in chosen]

            rows.append({
                "sample_id":  sample_id,
                "compound":   "+".join(chosen),
                "label":      str(component_labels),
                "is_mixture": True,
                "components": "+".join(chosen),
                "png_path":   png_path,
                "npy_path":   npy_path,
            })

        print(f"\nZmesi → {n_mixtures} vzoriek")

    # Ulož index
    index_df = pd.DataFrame(rows)
    index_df.to_csv(os.path.join(out_dir, "index.csv"), index=False)

    pure_count    = len([r for r in rows if not r["is_mixture"]])
    mixture_count = len([r for r in rows if r["is_mixture"]])

    print(f"\nDataset uložený do: {out_dir}")
    print(f"Čisté vzorky:  {pure_count}")
    print(f"Zmesi:         {mixture_count}")
    print(f"Celkovo:       {len(rows)}")
    print(f"Zlúčeniny ({len(compounds)}): {compounds}")


# ─────────────────────────────────────────────────────────────────────────────
# SPUSTENIE
# ─────────────────────────────────────────────────────────────────────────────

CSV_PATH = "../spectra_dataset.csv"
OUT_DIR  = "../nmr_dataset"

build_dataset(
    csv_path             = CSV_PATH,
    out_dir              = OUT_DIR,
    samples_per_compound = 5,
    save_vectors         = True,
    save_images          = True,
    vector_len           = 1024,
    generate_mixtures    = True,
    mixture_ratio        = 0.3,
    max_components       = 3,
    seed                 = 42,
)

Čisté: acetone → 3 vzoriek
Čisté: ethanol → 3 vzoriek
Čisté: benzene → 3 vzoriek
Čisté: toluene → 3 vzoriek
Čisté: chloroform → 3 vzoriek
Čisté: methanol → 3 vzoriek
Čisté: dimethyl_sulfoxide → 3 vzoriek
Čisté: acetic_acid → 3 vzoriek
Čisté: diethyl_ether → 3 vzoriek
Čisté: cyclohexane → 3 vzoriek
Čisté: hexane → 3 vzoriek
Čisté: dichloromethane → 3 vzoriek
Čisté: tetrahydrofuran → 3 vzoriek
Čisté: acetonitrile → 3 vzoriek
Čisté: pyridine → 3 vzoriek
Čisté: dimethylformamide → 3 vzoriek
Čisté: formic_acid → 3 vzoriek
Čisté: propanol → 3 vzoriek
Čisté: isopropanol → 3 vzoriek
Čisté: acetaldehyde → 3 vzoriek
Čisté: ethyl_acetate → 3 vzoriek
Čisté: propanoic_acid → 3 vzoriek
Čisté: nitromethane → 3 vzoriek
Čisté: dioxane → 3 vzoriek
Čisté: trimethylamine → 3 vzoriek
Čisté: glycerol → 3 vzoriek
Čisté: ethylene_glycol → 3 vzoriek
Čisté: acetophenone → 3 vzoriek
Čisté: benzaldehyde → 3 vzoriek
Čisté: aniline → 3 vzoriek
Čisté: phenol → 3 vzoriek
Čisté: diethylamine → 3 vzoriek
Čisté: triethy